# Day 046 Solution — Time Series Basics

Demonstrates: DatetimeIndex, resampling, rolling windows, change analysis, TrendAnalyzer class, and a saved matplotlib chart.
All data is generated in-cell — no external files required.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)


def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }


def rolling_mean(series: pd.Series, window: int, min_periods: int = 1) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).mean()

def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    r = series.rolling(window=window, min_periods=1)
    return pd.DataFrame({
        'mean': r.mean(),
        'std':  r.std(),
        'min':  r.min(),
        'max':  r.max(),
    })


def period_changes(series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        'value':      series,
        'change':     series.diff(),
        'pct_change': series.pct_change() * 100,
        'cumulative': series.cumsum(),
    })


class TrendAnalyzer:
    def __init__(self):
        self._series = None

    def load(self, df: pd.DataFrame, date_col: str, value_col: str) -> 'TrendAnalyzer':
        self._series = parse_time_series(df, date_col)[value_col].dropna()
        return self

    def summary(self) -> dict:
        s = self._series
        if s is None or len(s) == 0:
            return {}
        first, last = float(s.iloc[0]), float(s.iloc[-1])
        total_pct   = (last - first) / abs(first) * 100 if first != 0 else 0.0
        direction   = 'up' if total_pct > 1 else ('down' if total_pct < -1 else 'flat')
        return {
            'n_periods':        len(s),
            'start_date':       str(s.index[0].date()),
            'end_date':         str(s.index[-1].date()),
            'first_value':      round(first, 2),
            'last_value':       round(last, 2),
            'total_pct_change': round(total_pct, 2),
            'trend_direction':  direction,
            'weekly_avg':       resample_series(s, 'W', 'mean'),
            'rolling_mean_7d':  rolling_stats(s, 7)['mean'],
            'daily_changes':    period_changes(s),
        }

## Step 1 — Build a DatetimeIndex

In [ ]:
df_raw = make_sample_ts(n_days=90, seed=42)
df_ts  = parse_time_series(df_raw, 'date')
series = df_ts['value']

print(f'Shape:      {df_ts.shape}')
print(f'Index type: {type(df_ts.index).__name__}')
print(f'Date range: {series.index[0].date()} → {series.index[-1].date()}')

feats = date_features(df_ts)
print(f'\nFirst row features:')
print(feats.iloc[0].to_dict())

assert isinstance(df_ts.index, pd.DatetimeIndex)
assert df_ts.index.is_monotonic_increasing
assert list(feats.columns) == ['year', 'month', 'day', 'day_of_week', 'quarter']

## Step 2 — Resample at Multiple Frequencies

In [ ]:
summary = multi_freq_summary(series)
print(f'Daily  rows: {len(summary["daily"])}')
print(f'Weekly rows: {len(summary["weekly"])}')

monthly = resample_series(series, 'ME', 'mean')
print(f'Monthly rows: {len(monthly)}')
print(monthly.round(2))

assert len(summary['daily'])  == 90
assert len(summary['weekly'])  < 90

## Step 3 — Rolling Windows

In [ ]:
rm7 = rolling_mean(series, window=7)
rs7 = rolling_stats(series, window=7)

print(f'First rolling mean (w=7): {rm7.iloc[0]:.2f}  (single point, min_periods=1)')
print(f'7th rolling mean  (w=7):  {rm7.iloc[6]:.2f}  (full 7-day window)')
print(f'\nRolling stats (last 5 rows):')
print(rs7.tail().round(2))

assert len(rm7) == len(series)
assert not rm7.isna().any()

## Step 4 — Change Analysis

In [ ]:
changes = period_changes(series)
print('First 5 rows of period_changes:')
print(changes.head().round(2))

print(f'\nAverage daily change: {changes["change"].mean():.2f}')
print(f'Max single-day gain:  {changes["change"].max():.2f}')
print(f'Max single-day loss:  {changes["change"].min():.2f}')

assert pd.isna(changes['change'].iloc[0])
assert not pd.isna(changes['change'].iloc[1])

## Step 5 — TrendAnalyzer Full Report

In [ ]:
ta     = TrendAnalyzer()
report = ta.load(df_raw, 'date', 'value').summary()

print('=== Trend Report ===')
print(f"Periods:       {report['n_periods']}")
print(f"Range:         {report['start_date']} -> {report['end_date']}")
print(f"First/Last:    {report['first_value']:.2f} -> {report['last_value']:.2f}")
print(f"Total change:  {report['total_pct_change']:+.2f}%")
print(f"Direction:     {report['trend_direction']}")

assert report['n_periods']    == 90
assert report['start_date']   == '2024-01-01'
assert report['trend_direction'] in ('up', 'down', 'flat')

## Step 6 — Save Trend Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Raw + 7-day rolling mean
series.plot(ax=ax1, alpha=0.4, label='Daily value', color='steelblue')
report['rolling_mean_7d'].plot(ax=ax1, label='7-day rolling mean',
                               color='navy', linewidth=2)
ax1.set_title('Daily Values with 7-Day Rolling Mean')
ax1.set_ylabel('Value')
ax1.legend()

# Daily % change
report['daily_changes']['pct_change'].plot(ax=ax2, color='darkorange', alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_title('Daily % Change')
ax2.set_ylabel('% Change')
ax2.set_xlabel('Date')

plt.tight_layout()
fig.savefig('trend_chart.png', bbox_inches='tight', dpi=100)
plt.close('all')
print('Chart saved: trend_chart.png')

import os
assert os.path.exists('trend_chart.png') and os.path.getsize('trend_chart.png') > 1000

print('\nTime Series Basics complete!')